# Task 2: Variational autoencoder based Topic Model 

In [ ]:
%pip install pandas numpy nltk spacy scikit-learn torch

In [ ]:
import pandas as pd
import numpy as np
import nltk
nltk.download('stopwords')
from spacy.lang.en import English
nlp = English()
import torch
from torch.utils.data import DataLoader
from datetime import datetime
import os
import logging
from torch.utils.tensorboard import SummaryWriter
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.en.stop_words import STOP_WORDS
tokenizer = nlp.tokenizer
all_punctuations = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
all_punctuations = [token for token in all_punctuations]
stop_words = list(STOP_WORDS) + all_punctuations

## TODO: Import the data
Get ten year data from the downloaded file. 

In [ ]:
writer = SummaryWriter()

# Make the Data
print('reading raw data...')
# TODO: FIX THE PATH BELOW!
dataset_filename = "path/to/papers.csv"
data = pd.read_csv(dataset_filename, sep=',', usecols=['year', 'full_text'])

# TODO: get 10 years of data
# Hint: sort the data by year

# Let's see the first 5 rows
print(data.head(n=5))


## TODO: Preprocessing
Apply preprocessing steps (such as tokenization and removing stopwords)

In [ ]:
# preprocess data
def transform(texts):
    # TODO: tokenize the data
    texts = ...
    # lowercase the data
    texts = [text.lower() for text in texts]
    # TODO: remove stop words and non-alphabetic characters
    # Hint: use the stop_words above and the isalpha() method 
    texts = ...
    # remove short words
    texts = [text for text in texts if len(text) > 3]
    return texts

# Splitting the data
Separate your input data into training, validation, and testing subsets by running the cell below

In [ ]:
def text_data(data, train_idx, test_idx, val_idx, batch_size):
    docs = []
    for l in range(len(data)):
        print(l)
        doc = transform(str(data["full_text"][l]))
        doc = " ".join(doc)
        docs.append(doc)
    
    vectorizer = CountVectorizer(max_df=0.8, min_df=50, lowercase=True, stop_words='english')
    bow = vectorizer.fit_transform(docs)
    docs = torch.from_numpy(bow.toarray()).float()
    boww = vectorizer.inverse_transform(docs)
    texts = []
    for i in boww:
        texts.append(i.tolist())
    vocab = pd.DataFrame(columns=['word', 'index'])
    vocab['word'] =  sorted(vectorizer.vocabulary_.keys())
    vocab['index'] = vocab.index
    train_docs, val_docs, test_docs = docs[train_idx:], docs[-test_idx: -val_idx], docs[-val_idx: -1]
    train_dl = DataLoader([[train_docs[i]] for i in range(len(train_docs))], shuffle=True, batch_size=batch_size)
    val_dl = DataLoader([[val_docs[i]] for i in range(len(val_docs))], batch_size=batch_size)
    test_dl = DataLoader([[test_docs[i]] for i in range(len(test_docs))], batch_size=batch_size)
    return vocab, docs, train_dl, val_dl, test_dl, bow, texts

vocab, docs, train_dl, val_dl, test_dl, bow, texts = text_data(data, train_idx=0, test_idx=1000, val_idx=500, batch_size=32)
vocab_size = docs.shape[1]

## TODO: Implement Dirichlet VAE
Use the Gaussian VAE model model below and fill in the code to implement a Dirichlet VAE by using the Dirichlet distribution as a prior on the latent variables.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import LogNormal, Dirichlet, Gamma, Laplace
from torch.distributions import kl_divergence

class EncoderModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear_layer_one = nn.Linear(vocab_size, hidden_size[0])
        self.linear_layer_two = nn.Linear(hidden_size[0], hidden_size[1])
        self.linear_layer_three = nn.Linear(hidden_size[1], hidden_size[2])
        
    def forward(self, inputs):
        activation = nn.LeakyReLU()
        hidden_layer_one = activation(self.linear_layer_one(inputs))
        hidden_layer_two = self.dropout(activation(self.linear_layer_two(hidden_layer_one)))
        hidden_layer_three = self.dropout(activation(self.linear_layer_three(hidden_layer_two)))
        return hidden_layer_three


class DecoderModule(nn.Module):
    def __init__(self, vocab_size, num_topics, dropout):
        super().__init__()
        self.topics_to_doc = nn.Linear(num_topics, vocab_size)
        self.batch_normalization = nn.BatchNorm1d(vocab_size, affine=False)
        
    def forward(self, inputs):
        log_softmax = nn.LogSoftmax(dim = 1)
        return log_softmax(self.batch_normalization(self.topics_to_doc(inputs)))


class EncoderToLogNormal(nn.Module):
    def __init__(self, hidden_size, num_topics):
        super().__init__()
        self.linear_mean = nn.Linear(hidden_size[2], num_topics)
        self.linear_var = nn.Linear(hidden_size[2], num_topics)
        self.batch_norm_mean = nn.BatchNorm1d(num_topics, affine=False)
        self.batch_norm_var = nn.BatchNorm1d(num_topics, affine=False)
    
    def forward(self, hidden):
        mean = self.batch_norm_mean(self.linear_mean(hidden))
        var = 0.5 * self.batch_norm_var(self.linear_var(hidden))
        dist = LogNormal(mean, var.exp())
        return dist
        
class EncoderToDirichlet(nn.Module):
    def __init__(self, hidden_size, num_topics):
        super().__init__()
        self.linear_alpha = nn.Linear(hidden_size[2], num_topics)
        self.batch_norm_alpha = nn.BatchNorm1d(num_topics, affine=False)
    
    def forward(self, hidden):
        # TODO: Implement the Dirichlet distribution and return it
        alpha = ...
        dist = ...
        return dist


class VAE(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_topics, dropout, model_type, beta):
        super().__init__()
        self.encoder = EncoderModule(vocab_size, hidden_size, dropout)
        if model_type == 1:
            self.encoder_to_dist = EncoderToLogNormal(hidden_size, num_topics)
        elif model_type == 2:
            self.encoder_to_dist = EncoderToDirichlet(hidden_size, num_topics)
        self.decoder = DecoderModule(vocab_size, num_topics, dropout)
        self.beta = beta
        
    def forward(self, inputs):
        encoder_output = self.encoder(inputs)
        dist = self.encoder_to_dist(encoder_output)
        if self.training:
            dist_to_decoder = dist.rsample().to(inputs.device)
        else:
            dist_to_decoder = dist.mean.to(inputs.device)
        softmax = nn.Softmax(dim = 1)
        dist_to_decoder = softmax(dist_to_decoder)
        reconstructed_documents = self.decoder(dist_to_decoder)
        return reconstructed_documents, dist
    
    def loss(self, reconstructed, original, posterior): # We need to have NLL Loss as well KLD Loss
        if isinstance(posterior, LogNormal):
            loc = torch.zeros_like(posterior.loc)
            scale = torch.ones_like(posterior.scale)        
            prior = LogNormal(loc, scale)
        
        elif isinstance(posterior, Dirichlet):
            # TODO: implement the Dirichlet prior 
            alpha = 0.1
            alphas = ...
            prior = ...

        NLL = - torch.sum(reconstructed*original)
        KLD = torch.sum(kl_divergence(posterior, prior).to(reconstructed.device))
        loss_for_training = NLL + self.beta * KLD
        return NLL, KLD, loss_for_training

## Train the Dirichlet VAE Model
Useful functions to train the Dirichlet VAE Model.

In [25]:
def train_one_epoch(train_dl, model, optim, device):
    model.train()
    epoch_total_loss, epoch_nll_loss, epoch_kld_loss = [], [], []
    for batch in train_dl:
        batch_total_loss, batch_nll_loss, batch_kld_loss = train_one_batch(batch, model, optim, device)
        epoch_total_loss.append(batch_total_loss), epoch_nll_loss.append(batch_nll_loss), epoch_kld_loss.append(batch_kld_loss)
    loss ={'total_loss': torch.mean(torch.Tensor(epoch_total_loss)), 'nll_loss': torch.mean(torch.Tensor(epoch_nll_loss)), 'kld_loss': torch.mean(torch.Tensor(epoch_kld_loss))}
    return loss

def train_one_batch(batch, model, optim, device):
    docs = batch[0].to(torch.device(device))
    optim.zero_grad()
    out, posterior = model(docs)
    nll, kld, loss_for_training = model.loss(out, docs, posterior)
    loss = nll + kld
    loss_for_training.backward()
    optim.step()
    return loss.item(), nll.item(), kld.item()

def validate_one_epoch(val_dl, model, device):
    model.eval()
    epoch_total_loss, epoch_nll_loss, epoch_kld_loss = [], [], []
    for batch in val_dl:
        batch_total_loss, batch_nll_loss, batch_kld_loss = validate_one_batch(batch, model, device)
        epoch_total_loss.append(batch_total_loss), epoch_nll_loss.append(batch_nll_loss), epoch_kld_loss.append(batch_kld_loss)
    loss ={'total_loss': torch.mean(torch.Tensor(epoch_total_loss)), 'nll_loss': torch.mean(torch.Tensor(epoch_nll_loss)), 'kld_loss': torch.mean(torch.Tensor(epoch_kld_loss))}
    return loss

def validate_one_batch(batch, model, device):
    docs = batch[0].to(torch.device(device))
    out, posterior = model(docs)
    nll, kld, _ = model.loss(out, docs, posterior)
    loss = nll + kld
    return loss.item(), nll.item(), kld.item()

def fit(epochs, train_dl, val_dl, model, optim, device, path, writer):
    history = []
    for epoch in range(epochs):
        epoch_train_loss = train_one_epoch(train_dl, model, optim, device)
        epoch_validation_loss = validate_one_epoch(val_dl, model, device)
        writer.add_scalar("Loss/train", epoch_train_loss['total_loss'], epoch)
        writer.add_scalar("Loss/eval", epoch_validation_loss['total_loss'], epoch)
        log = {
                'epoch': epoch + 1,
                'train_loss': epoch_train_loss['total_loss'],
                'train_loss_nll': epoch_train_loss['nll_loss'],
                'train_loss_kld': epoch_train_loss['kld_loss'],
            }
        history.append(log)
        print(log)
    beta = model.decoder.topics_to_doc.weight.cpu().detach().T
    return beta, history

Run the cell below to train the model with 25 topics.

In [ ]:
print('vocab_size :', vocab_size)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(3)
hidden_size = [512, 256, 128]
num_topics = 25
dropout = 0.2
epochs = 10

# Declaring model and optimizer
model = VAE(vocab_size, hidden_size, num_topics, dropout, 2, 3)
model = model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

#  Making folder for storing outputs
now = datetime.now()
path = now.strftime("outputs/%d.%m.%y.%H.%M.%S.") + "test"
os.makedirs(path)

# Run training
beta, history = fit(epochs, train_dl, val_dl, model, optim, device, path, writer)

In [ ]:
# Save the ten top words for each topic in a file
def plot(beta, vocab, path):
    filename = path +'/topics.txt'
    f= open(filename,"w+")
    for i in range(beta.shape[0]):
        sorted_, indices = torch.sort(beta[i], descending=True)
        df = pd.DataFrame(indices[:10].numpy(), columns=['index'])
        names = pd.merge(df, vocab[['index', 'word']], how='left', on='index')['word'].values
        f.write(' '.join(names))
        f.write('\n')
        print(' '.join(names))
    f.close()

plot(beta, vocab, path)


## TODO: Get 25 topics using Dirichlet VAE model with three different values for $\alpha\in\{0.1,1.0,10.0\}$.

Repeat the process above for different values of $\alpha$.